In [1]:
import pandas as pd
import numpy as np
import mssql_python
from pathlib import Path


In [2]:
# Data path
base_path = Path('C:/Users/User/OneDrive/Desktop/UN')
data_path = Path('C:/Users/User/OneDrive/Desktop/UN/Data')
data_path = Path('C:/Users/GuestUser/Desktop/UN/Data')


data_files_dict = {'index_1030' : ['index_1030 Gender Development Index.xlsx', '2025_metadata_Gender Development Index 2025.xlsx'],
                   'index_9' : ['index_9 EGDI.xlsx', '20251102_metadata_E-Government Development Index_2024.xlsx', 'raw_EGDI_2024.xlsx'],
                   'index_1032': ['20260710_historical data_index_1032_SDG 2026.xlsx', '20260710_metadata_Sustainable_Development_Index_2026.xlsx', 'raw_SDG_2026.xlsx'],

                    # Somalia indices
                   'index_12' : ['index_12 E-participation Index.xlsx', '', 'raw_EPI_2024.xlsx'],
                   'index_1049' : ['index_1049.xlsx', '', 'raw_IIAG_2024.xlsx'],
                   'index_1037' : ['index_1037.xlsx', '', 'raw_GOVTECH_2022.xlsx'],
                   'index_1022': ['index_1022 Women Peace and security Index.xlsx','','raw_WPS_2025.xlsx'],
                   
                   # Somalia second batch
                   'index_1050': ['index_1050 Global Hunger.xlsx','','raw_GHI.xlsx'],
                   'index_5': ['index_5 ICT Developement Index.xlsx','','raw_ITU_2024.xlsx'],
                   'index_1036': ['20262205_historical data_index_1036_WPF_2026.xlsx','','raw_WPF_2026.xlsx'],

                   # Third
                   'index_1027': ['index_1027 Human Development Index.xlsx','','raw_HDI.xlsx'],
                   'index_1030': ['index_1030 Gender Development Index.xlsx','','raw_GDI_2024.xlsx'],
                   'index_1031': ['index_1031 Planetary adjusted HDI.xlsx','','raw_PPHDI.xlsx'],
                   'index_1035': ['20260303_Historical data _WBL2.0_2026.xlsx','','raw_WBL2.0_2026.xlsx'],
                   # None,
                   'index_1026': ['index_1026 Open data Policies.xlsx','','raw_ODP_2022.xlsx'],
                   'index_1053': ['20260721_historical data_index_1053_BTI2026.xlsx','','raw_BTI_2026.xlsx'],
                   'index_1021': ['index_1021.xlsx','','raw_WBL 1.0.xlsx'],
                   'index_1052': ['20260717_historical data_index_1052_WPJ2026.xlsx','','raw_WJP_2025.xlsx'],
                   'index_1034': ['index_1034.xlsx','','raw_EnviPI_2026.xlsx'],
            
                  }
#Human Development Index (index_1027)
#Gender Development Index (index_1030)
#Planetary adjusted Human development Index (index_1031)
#Women Business and the Law 2.0 (index_1035)

index_name = 'index_9'
index_data_files = data_files_dict[index_name]

index_historical_kpi_data_path = Path(data_path) / 'Historical' / index_data_files[0]
#index_metadata_path = Path(data_path) / 'Metadata' / index_data_files[1]
index_simulator_data_path = Path(data_path) / 'Simulator Data' / index_data_files[2]
diagnosis_assessment_template_path = Path(data_path) / 'Reference templates' / 'Data_Diagnosis_Assessment_template.xlsx'

# Reference template
#diagnosis_assessment_template = pd.read_excel(diagnosis_assessment_template_path)

# Read raw data
historical_kpi_data_raw = pd.read_excel(index_historical_kpi_data_path)
#index_metadata_raw = pd.read_excel(index_metadata_path)
#index_metadata_raw['Arab Countries'] = index_metadata_raw['Arab Countries'].str.strip()
simulator_data_raw = pd.read_excel(index_simulator_data_path)

country_iso = pd.read_excel(Path(data_path) / 'metadata_SDG_2026.xlsx', sheet_name='country')
country_iso['country'] = country_iso['country'].str.strip()
country_iso['ISO'] = country_iso['ISO'].str.strip()
# - data correction
if index_name == 'index_1032':
    #index_metadata_raw['Index_ID'] = '1032'
    #index_metadata_raw = index_metadata_raw.merge(country_iso, how='left', left_on='Arab Countries', right_on='country')
    # make first row columns names
    historical_kpi_data_raw.columns = historical_kpi_data_raw.iloc[0]
    #index_data_raw = index_data_raw[1:]


# Reference Data
escwa_members = ['DZA','BHR','COM','DJI','EGY','IRQ','JOR','KWT','LBN','LBY','MRT',
                 'MAR','OMN','PSE','QAT','SAU','SOM','SDN','SYR','TUN','ARE','YEM']


kpi_key = (simulator_data_raw.iloc[0]
           .rename('Series Name')
           .rename_axis('KPI ID')
           .reset_index()
           .dropna(subset=['Series Name']))
kpi_key['Series Name'] = kpi_key['Series Name'].astype(str).str.strip()
kpi_key['KPI ID'] = kpi_key['KPI ID'].astype(str).str.strip()
######## Sim data ###################
simulator_data = simulator_data_raw.copy()
simulator_data = simulator_data[1:].reset_index(drop=True)

pivot_names = ['Country', 'ISO', 'Region', 'Sub-region', 'Income Type']
# Convert to long format
simulator_data = simulator_data.melt(id_vars=pivot_names, 
                         var_name='KPI ID', value_name='data_report')      # convert wide to long
simulator_data['KPI ID'] = simulator_data['KPI ID'].astype(str).str.strip()
simulator_data = simulator_data.merge(kpi_key,how='left')
simulator_data = simulator_data[['Country', 'ISO', 'Series Name', 'KPI ID', 'data_report']]
simulator_data['data_report'] = pd.to_numeric(
    simulator_data['data_report'], errors='coerce'
)
########## New ############


# Index data minor reformatting and filtering
historical_kpi_data = historical_kpi_data_raw.copy()
historical_kpi_data = historical_kpi_data.rename(columns={'Unnamed: 0':'KPI ID','Unnamed: 1':'Series Name','Unnamed: 2':'year_historical_automated',
                                        'year':'year_historical_automated', 'Series name':'Series Name', 'KeyID':'KPI ID'})
historical_kpi_data = historical_kpi_data.iloc[1:]
# - Values to change, match simulator values
replacement_dict = {
    'Mean years of schooling': 'Mean Year of Schooling',
    'Expected years of schooling': 'Expected Year of Schooling'
}
historical_kpi_data['Series Name'] = historical_kpi_data['Series Name'].replace(replacement_dict)
# - long format
historical_kpi_data_long = historical_kpi_data.melt(id_vars=['KPI ID','Series Name','year_historical_automated'], 
                                           var_name='ISO', value_name='data_historical_automated')        # convert wide to long
#historical_kpi_data_long = historical_kpi_data_long[historical_kpi_data_long['ISO'].isin(escwa_members)]
# - format values for consistency
historical_kpi_data_long['data_historical_automated'] = pd.to_numeric(
    historical_kpi_data_long['data_historical_automated'], errors='coerce').astype('Float64')
historical_kpi_data_long['KPI ID'] = historical_kpi_data_long['KPI ID'].astype(str).str.strip()
historical_kpi_data_long['year_historical_automated'] = historical_kpi_data_long['year_historical_automated'].astype('Int64')

series_to_kpi = historical_kpi_data_long[['Series Name', 'KPI ID']].copy().drop_duplicates().reset_index(drop=True)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\GuestUser\\Desktop\\UN\\Data\\Historical\\index_9 EGDI.xlsx'

In [ ]:
kpi_list = simulator_data['KPI ID'].unique().tolist()

In [3]:
# Prod connection test
server_name='sqldb-p-server-001.database.windows.net'
database_name='ispar'
username='ispar_user_read'
pwd='TAescwa2025##'

connection_string = (
    f"Server={server_name};"
    f"Database={database_name};"
    f"UID={username};"
    f"PWD={pwd};"
    "TrustServerCertificate=Yes"
)

# Establish connection
with mssql_python.connect(connection_string) as conn:
    # Create a cursor object
    with conn.cursor() as cursor:
        # Execute statement to look up user tables
        cursor.execute(
            """
            SELECT TABLE_SCHEMA, TABLE_NAME 
            FROM INFORMATION_SCHEMA.TABLES 
            WHERE TABLE_TYPE = 'BASE TABLE';
            """
        )
        # Fetch and print table names
        for row in cursor.fetchall():
            print(f"Schema: {row[0]}, Table: {row[1]}")
            

Schema: dbo, Table: PublicAdministrationKPIs
Schema: dbo, Table: CountryRegionalCommission
Schema: dbo, Table: Benchmarks
Schema: dbo, Table: Data_SCHEMA_CHANGE_LOG
Schema: dbo, Table: Newsletter_EventThemes
Schema: dbo, Table: Pages_PageAccessRequests
Schema: dbo, Table: Policies
Schema: dbo, Table: AspNetUsers
Schema: dbo, Table: Data_HistoricalKPIChange
Schema: dbo, Table: PoliciesCategories
Schema: dbo, Table: Data_NOTIFICATION_LOG
Schema: dbo, Table: AspNetUserRoles
Schema: dbo, Table: Newsletter_EventTypes
Schema: dbo, Table: Data_CountryHistoricalChange
Schema: dbo, Table: Pages
Schema: dbo, Table: PoliciesByKPIs
Schema: dbo, Table: Data_HistoricalKPI_BackUp
Schema: dbo, Table: AspNetUserClaims
Schema: dbo, Table: Data_HASH_TRACKING
Schema: dbo, Table: KPIData
Schema: dbo, Table: KPIThemeAssignment
Schema: dbo, Table: Data_KPIsfiles
Schema: dbo, Table: ExternalPopulation
Schema: dbo, Table: HistoricalKPI_backup
Schema: dbo, Table: AspNetUserLogins
Schema: dbo, Table: Simulation_

In [6]:

server_name='escwa-dev-02.escwa.un.org'
database_name='ispar'
username='ispar'
pwd='ispar_data'

connection_string = (
    f"Server={server_name};"
    f"Database={database_name};"
    f"UID={username};"
    f"PWD={pwd};"
    "TrustServerCertificate=Yes"
)

# Establish connection
with mssql_python.connect(connection_string) as conn:
    # Create a cursor object
    with conn.cursor() as cursor:
        # Execute statement to look up user tables
        cursor.execute(
            """
            SELECT TABLE_SCHEMA, TABLE_NAME 
            FROM INFORMATION_SCHEMA.TABLES 
            WHERE TABLE_TYPE = 'BASE TABLE';
            """
        )
        # Fetch and print table names
        for row in cursor.fetchall():
            print(f"Schema: {row[0]}, Table: {row[1]}")
            

Schema: dbo, Table: GenderCountryPopulation
Schema: dbo, Table: HistoricalKPI
Schema: dbo, Table: HistoricalKPI_240517
Schema: dbo, Table: HistoricalKPI_backup
Schema: dbo, Table: HistoricalKPIChange
Schema: dbo, Table: HistoricalKPIChange_240517
Schema: dbo, Table: HistoricalPeriods
Schema: dbo, Table: LNOBRegionRank
Schema: dbo, Table: Indexes_Not_On_Ispar
Schema: dbo, Table: LNOBGlobalRank
Schema: dbo, Table: Indexes_Not_On_Ispar_Data
Schema: dbo, Table: IndexImprovement
Schema: dbo, Table: IndexMax
Schema: dbo, Table: sysdiagrams
Schema: dbo, Table: KPI_Extended_Metadata
Schema: dbo, Table: temp_caps
Schema: dbo, Table: KPIHistoryAvailable
Schema: dbo, Table: KPIPolicies
Schema: dbo, Table: KPIs_b
Schema: dbo, Table: KPIs_Custom_Min_Max
Schema: dbo, Table: KPIs_test
Schema: dbo, Table: KPIsStructureType
Schema: dbo, Table: LinkBetweenPillarsPol
Schema: dbo, Table: Logs
Schema: dbo, Table: MappedKPIs
Schema: dbo, Table: KPIData_Rank_Test
Schema: dbo, Table: MappedKPIsByGender
Schema

In [57]:
kpi_list

['1237',
 '1238',
 '1239',
 '1240',
 '1241',
 '271',
 '272',
 '1942',
 '1941',
 '274',
 '279',
 '280',
 '281',
 '282',
 '1940']

In [ ]:
import mssql_python

server_name = 'escwa-dev-02.escwa.un.org'
database_name = 'ispar'
username = 'ispar'
pwd = 'ispar_data'  # pull from env, e.g. os.environ['ISPAR_PWD']

connection_string = (
    f"Server={server_name};"
    f"Database={database_name};"
    f"UID={username};"
    f"PWD={pwd};"
    "TrustServerCertificate=Yes"
)


In [34]:
# KPI query
placeholders = ",".join("?" for _ in kpi_list)
hist_data_query = f"SELECT * FROM HistoricalKPI WHERE KPIID IN ({placeholders});"

with mssql_python.connect(connection_string) as conn:
    with conn.cursor() as cursor:
        cursor.execute(hist_data_query, kpi_list)
        columns = [col[0] for col in cursor.description]
        rows = cursor.fetchall()

hist_data = pd.DataFrame.from_records(rows, columns=columns)

In [35]:
country_ref_tbl_query = "SELECT keyID,Name,ISO FROM Countries;"
with mssql_python.connect(connection_string) as conn:
    with conn.cursor() as cursor:
        cursor.execute(country_ref_tbl_query)
        columns = [col[0] for col in cursor.description]
        rows = cursor.fetchall()
country_ref_tbl = pd.DataFrame.from_records(rows, columns=columns)
country_ref_tbl = country_ref_tbl.rename(columns={'keyID':'CountryID'})

In [36]:
hist_data = hist_data.merge(country_ref_tbl).add_suffix("_db")
hist_data['Value_db'] = pd.to_numeric(hist_data['Value_db'], errors='coerce').astype('Float64')
hist_data['KPIID_db'] = hist_data['KPIID_db'].astype(str).str.strip()
hist_data['Year_db'] = hist_data['Year_db'].astype('Int64')


In [37]:
historical_kpi_data_long

,KPI ID,Series Name,year_historical_automated,ISO,data_historical_automated
0,282,Mean Year of Schooling,2022,AFG,3.0
1,282,Mean Year of Schooling,1990,AFG,1.5
2,282,Mean Year of Schooling,1991,AFG,1.6
3,282,Mean Year of Schooling,1992,AFG,1.6
4,282,Mean Year of Schooling,1993,AFG,1.7
...,...,...,...,...,...
56827,274,Active mobile-broadband subscriptions per 100 ...,2020,ZWE,56.0
56828,274,Active mobile-broadband subscriptions per 100 ...,2021,ZWE,59.1
56829,274,Active mobile-broadband subscriptions per 100 ...,2022,ZWE,60.5
56830,274,Active mobile-broadband subscriptions per 100 ...,2023,ZWE,67.6


In [47]:
print(historical_kpi_data_long.shape)
joined = historical_kpi_data_long.merge(hist_data, how='left',
                                        left_on=['KPI ID', 'year_historical_automated','ISO'],
                                        right_on=['KPIID_db','Year_db','ISO_db'])

print(joined.shape)
joined = joined[joined['Value_db'].notna()]
print(joined.shape)
joined['diff'] = abs(joined['Value_db'] - joined['data_historical_automated'])


(56832, 5)
(56832, 12)
(39325, 12)


In [49]:
joined['diff']

0        0.0
1        0.0
2        0.0
3        0.0
4        0.0
        ... 
56827    0.0
56828    0.0
56829    0.0
56830    0.0
56831    0.0
Name: diff, Length: 39325, dtype: Float64

In [60]:
t = joined[joined['diff'] > 0.00001]

In [61]:
t.to_clipboard()

In [32]:
historical_kpi_data_long['data_historical_automated'] = pd.to_numeric(
    historical_kpi_data_long['data_historical_automated'], errors='coerce').astype('Float64')
historical_kpi_data_long['KPI ID'] = historical_kpi_data_long['KPI ID'].astype(str).str.strip()
historical_kpi_data_long['year_historical_automated'] = historical_kpi_data_long['year_historical_automated'].astype('Int64')


,KPIID_db,CountryID_db,Year_db,Category_db,Value_db,Name_db,ISO_db
0,271,1,1990,1,0.00,Afghanistan,AFG
1,271,1,1991,1,0.00,Afghanistan,AFG
2,271,1,1992,1,0.00,Afghanistan,AFG
3,271,1,1993,1,0.00,Afghanistan,AFG
4,271,1,1994,1,0.00,Afghanistan,AFG
...,...,...,...,...,...,...,...
39490,1942,199,2020,1,3.35,North Macedonia,MKD
39491,1942,199,2021,1,3.52,North Macedonia,MKD
39492,1942,199,2022,1,3.23,North Macedonia,MKD
39493,1942,199,2023,1,2.92,North Macedonia,MKD


In [14]:
import mssql_python

server_name = 'escwa-dev-02.escwa.un.org'
database_name = 'ispar'
username = 'ispar'
pwd = 'ispar_data'  # pull from env, e.g. os.environ['ISPAR_PWD']

connection_string = (
    f"Server={server_name};"
    f"Database={database_name};"
    f"UID={username};"
    f"PWD={pwd};"
    "TrustServerCertificate=Yes"
)



In [49]:
kpi_info.to_clipboard()

In [16]:
countries

,keyID,Name,ISO
0,1,Afghanistan,AFG
1,2,Albania,ALB
2,3,Algeria,DZA
3,4,Andorra,AND
4,5,Angola,AGO
...,...,...,...
198,207,Kosovo,KSV
199,208,Puerto Rico,PRI
200,209,Taiwan,TWN
201,211,Montserrat,MSR
